#### HAECA (Dobner et al.), Figure S6
##### GRN analysis using decoupler on spatial data
##### following: https://decoupler.readthedocs.io/en/latest/notebooks/spatial/rna_visium.html
##### data from E-MTAB-13084, Ganier et al. (DOI: 10.1073/pnas.2313326120), downloaded directly from https://cellxgene.cziscience.com/collections/34f12de7-c5e5-4813-a136-832677f98ac8 (all healthy (normal) samples)

In [ ]:
import os
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import stats

import scanpy as sc
import decoupler as dc
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib_scalebar.scalebar import ScaleBar

warnings.filterwarnings("ignore", category=FutureWarning)

#plot settings (make smaller)
sc.settings.verbosity = 2
sc.settings.autoshow = False
plt.rcParams['figure.figsize'] = (4, 3)
plt.rcParams['figure.dpi'] = 100

##### Load visium and deconvoluted, annotated anndata

In [ ]:
workdir = '~/HAECA/validation_datasets/E-MTAB-13084'
os.chdir(workdir)
datadir = os.path.join(workdir, 'cellxgene_datasets')
refdir  = os.path.join(workdir, 'raw_annotated')
outputdir = os.path.join(workdir, 'decoupler_output')
os.makedirs(outputdir, exist_ok=True)

#define genes of interest for plotting
TF_GENES = ['YAP1', 'TEAD1', 'TEAD4', 'WWTR1']

#set color pallettes for vascular beds and age brackets
EC_SUBTYPES = ['artery', 'capillary', 'lymphatic', 'vein']
PALETTE_SUBTYPES = {
    'artery': '#87092b', 'capillary': '#8cd1a9',
    'vein': '#269feb', 'lymphatic': '#cfca42', 'Other': 'lightgray'
}
PALETTE_EC_BINARY = {'Endothelial': 'red', 'Other': 'lightgray'}

AGE_GROUPS = ['Young (≤45)', 'Middle (46–60)', 'Aged (>60)']
GROUP_COLORS = {
    'Young (≤45)': '#4b4e52', 'Middle (46–60)': '#266c91', 'Aged (>60)': '#f0cf16'
}

def age_group(age):
    if age <= 45:  return 'Young (≤45)'
    elif age <= 60: return 'Middle (46–60)'
    else:           return 'Aged (>60)'

##### load metadata table

In [3]:
#metadata is missing as annotated visium was made from raw data (samples are WSSKNKCLsp-code) and the other is loaded from cellxgene using sample_id body_abdomen_A etc.
#load additional metadata
md = pd.read_csv("E-MTAB-13084.sdrf.txt", sep="\t")

md.head()
#this table contains "Source Name" (WSSKNKCLspnnnnnn) --> same as adata.obs['sample']
#contains "Characteristics[sample id]" --> e.g. face_temple1a

,Source Name,Comment[ENA_SAMPLE],Comment[BioSD_SAMPLE],Characteristics[organism],Characteristics[age],Unit[time unit],Term Source REF,Term Accession Number,Characteristics[developmental stage],Characteristics[sex],...,Comment[ENA_RUN],Comment[FASTQ_URI],Comment[read_type],Comment[read_index],Protocol REF.4,Derived Array Data File,Protocol REF.5,Derived Array Data File.1,Factor Value[sampling site],Factor Value[disease]
0,WSSKNKCLsp10446613,ERS15895820,SAMEA113901651,Homo sapiens,68,year,EFO,UO_0000036,adult,male,...,ERR11580418,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR115/ERR115...,sample_barcode,index1,P-MTAB-133975,WSSKNKCLsp10446613.jpg,P-MTAB-133975,WSSKNKCLsp10446613.tar.gz,temple,normal
1,WSSKNKCLsp10446613,ERS15895820,SAMEA113901651,Homo sapiens,68,year,EFO,UO_0000036,adult,male,...,ERR11580418,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR115/ERR115...,"spatial_barcode,umi_barcode",read1,P-MTAB-133975,WSSKNKCLsp10446613.jpg,P-MTAB-133975,WSSKNKCLsp10446613.tar.gz,temple,normal
2,WSSKNKCLsp10446613,ERS15895820,SAMEA113901651,Homo sapiens,68,year,EFO,UO_0000036,adult,male,...,ERR11580418,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR115/ERR115...,single,read2,P-MTAB-133975,WSSKNKCLsp10446613.jpg,P-MTAB-133975,WSSKNKCLsp10446613.tar.gz,temple,normal
3,WSSKNKCLsp10446614,ERS15895821,SAMEA113901652,Homo sapiens,68,year,EFO,UO_0000036,adult,male,...,ERR11580379,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR115/ERR115...,sample_barcode,index1,P-MTAB-133975,WSSKNKCLsp10446614.jpg,P-MTAB-133975,WSSKNKCLsp10446614.tar.gz,temple,normal
4,WSSKNKCLsp10446614,ERS15895821,SAMEA113901652,Homo sapiens,68,year,EFO,UO_0000036,adult,male,...,ERR11580379,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR115/ERR115...,"spatial_barcode,umi_barcode",read1,P-MTAB-133975,WSSKNKCLsp10446614.jpg,P-MTAB-133975,WSSKNKCLsp10446614.tar.gz,temple,normal


#### downloaded version misses some information on sampleID
##### created .csv from table above with matching sampleid and re-loaded here:

In [4]:
#load additional metadata that matches sample_id with donor_id:
metadata_to_add = pd.read_csv("metadata_sampleid.csv")
metadata_to_add.head()
#contains sample_id (matches with Source Name)
#contains Characteristics[sample id] (matches same column)
#also contains donor_id --> this matches the annotation in the visium data loaded below

# Rename columns in metadata
metadata_to_add = metadata_to_add.rename(columns={
    "Characteristics[disease]": "disease",
    "Characteristics[sample id]": "sample_id_2"
})

metadata_to_add.head()

,donor_id,sample_id_2,disease,sample_id
0,body_abdomen_A,body_abdomen1b,normal,WSSKNKCLsp10767965
1,body_back_A_a,body_back1a,normal,WSSKNKCLsp10446621
2,body_back_A_b,body_back1b,normal,WSSKNKCLsp12887264
3,body_breast_A,body_breast1,normal,WSSKNKCLsp12887263
4,body_inguinal_A_a,body_inguinal1a,normal,WSSKNKCLsp10446623


#### load reference anndatata

In [5]:
adata = sc.read_h5ad(os.path.join(refdir, "annotated_visium_adata.h5ad"))
adata.obs = adata.obs.rename(columns={"sample": "sample_id"})

lookup = metadata_to_add.drop_duplicates("sample_id").set_index("sample_id")
for col in ["donor_id", "disease", "sample_id_2"]:
    adata.obs[col] = adata.obs["sample_id"].map(lookup[col])

#keep only normal tissue (remove BCC cancer sampels)
adata = adata[adata.obs["disease"] == "normal"].copy()
print(adata.obs[['donor_id', 'sample_id', 'sample_id_2']].drop_duplicates())

                                                donor_id           sample_id  \
AAACACCAATAACTGC-1_WSSKNKCLsp10446621      body_back_A_a  WSSKNKCLsp10446621   
AAACAAGTATCTCCCA-1_WSSKNKCLsp12887266       face_scalp_A  WSSKNKCLsp12887266   
AAACAAGTATCTCCCA-1_WSSKNKCLsp10446623  body_inguinal_A_a  WSSKNKCLsp10446623   
AAACCCGAACGAAATC-1_WSSKNKCLsp12140369       face_cheek_B  WSSKNKCLsp12140369   
AAACAGCTTTCAGAAG-1_WSSKNKCLsp10446613      face_temple_A  WSSKNKCLsp10446613   
AAACAAGTATCTCCCA-1_WSSKNKCLsp10767968    body_inguinal_B  WSSKNKCLsp10767968   
AAACAAGTATCTCCCA-1_WSSKNKCLsp12887264      body_back_A_b  WSSKNKCLsp12887264   
AAACAAGTATCTCCCA-1_WSSKNKCLsp10767965     body_abdomen_A  WSSKNKCLsp10767965   
AAACAGCTTTCAGAAG-1_WSSKNKCLsp10446619  face_forehead_A_a  WSSKNKCLsp10446619   
AAACAGAGCGACTCCT-1_WSSKNKCLsp10767967       body_pubis_A  WSSKNKCLsp10767967   
AAACATTTCCCGGATT-1_WSSKNKCLsp12140367  face_forehead_A_b  WSSKNKCLsp12140367   
AAACAAGTATCTCCCA-1_WSSKNKCLsp10446617   

### load visium datasets

In [6]:
valid_donors = set(adata.obs['donor_id'].unique())

adatas = []
for f in sorted(os.listdir(datadir)):
    if not f.endswith('.h5ad'):
        continue
    sample_name = Path(f).stem
    if sample_name not in valid_donors:
        print(f'Skipped {f}: not in reference')
        continue

    ad = sc.read_h5ad(os.path.join(datadir, f))
    ad.obs_names_make_unique()
    ad.obs['sample_name'] = sample_name
    ad.obs['filename'] = f
    ad.obs['age'] = ad.obs['development_stage'].str.extract(r'(\d+)')[0]
    ad.obs['barcode'] = ad.obs_names.astype(str)
    ad.obs_names = ad.obs['barcode'] + "_" + ad.obs['sample_name']
    adatas.append(ad)
    print(f'Loaded {f}: {ad.shape}')

print(f"\nLoaded {len(adatas)} samples")

Loaded body_abdomen_A.h5ad: (4992, 35477)
Loaded body_back_A_a.h5ad: (4992, 35477)
Loaded body_back_A_b.h5ad: (4992, 35477)
Loaded body_breast_A.h5ad: (4992, 35477)
Loaded body_inguinal_A_a.h5ad: (4992, 35477)
Loaded body_inguinal_B.h5ad: (4992, 35477)
Loaded body_pubis_A.h5ad: (4992, 35477)
Loaded body_thigh_A.h5ad: (4992, 35477)
Loaded face_cheek_A.h5ad: (4992, 35477)
Loaded face_cheek_B.h5ad: (4992, 35477)
Loaded face_forehead_A_a.h5ad: (4992, 35477)
Loaded face_forehead_A_b.h5ad: (4992, 35477)
Loaded face_forehead_B.h5ad: (4992, 35477)
Loaded face_glabella_A.h5ad: (4992, 35477)
Loaded face_nose_A_a.h5ad: (4992, 35477)
Loaded face_nose_A_b.h5ad: (4992, 35477)
Loaded face_scalp_A.h5ad: (4992, 35477)
Loaded face_temple_A.h5ad: (4992, 35477)
Loaded face_temple_A_a.h5ad: (4992, 35477)
Loaded face_temple_B_a.h5ad: (4992, 35477)
Loaded face_temple_B_b.h5ad: (4992, 35477)

Loaded 21 samples


In [ ]:
cols_to_add = [
    'sample_id', 'sample_id_2', 'donor_id',
    'stromal', 'artery', 'capillary', 'lymphatic', 'vein', 'total_EC',
    'artery_frac', 'capillary_frac', 'lymphatic_frac', 'vein_frac', 'majority'
]

for ad in adatas:
    sample_name = ad.obs['sample_name'].iloc[0]
    ref_donor = (adata.obs[adata.obs['donor_id'] == sample_name]
                 .set_index('barcode')[cols_to_add])
    for col in cols_to_add:
        ad.obs[col] = ad.obs['barcode'].map(ref_donor[col])
    n = ad.obs['sample_id'].notna().sum()
    print(f"{sample_name}: {n}/{ad.n_obs} spots matched")

body_abdomen_A: 1755/4992 spots matched
body_back_A_a: 2418/4992 spots matched
body_back_A_b: 2546/4992 spots matched
body_breast_A: 2097/4992 spots matched
body_inguinal_A_a: 1310/4992 spots matched
body_inguinal_B: 1958/4992 spots matched
body_pubis_A: 1632/4992 spots matched
body_thigh_A: 1826/4992 spots matched
face_cheek_A: 2036/4992 spots matched
face_cheek_B: 1383/4992 spots matched
face_forehead_A_a: 1163/4992 spots matched
face_forehead_A_b: 963/4992 spots matched
face_forehead_B: 1987/4992 spots matched
face_glabella_A: 1577/4992 spots matched
face_nose_A_a: 1552/4992 spots matched
face_nose_A_b: 1642/4992 spots matched
face_scalp_A: 1344/4992 spots matched
face_temple_A: 986/4992 spots matched
face_temple_A_a: 927/4992 spots matched
face_temple_B_a: 1547/4992 spots matched
face_temple_B_b: 1494/4992 spots matched


##### preprocessing: filter for tissue spots, QC, normalize & log

In [13]:
def preprocess_visium(ad):
    ad = ad.copy()
    ad.var["ensembl_id"] = ad.var_names.astype(str)
    if "raw_counts" not in ad.layers:
        if ad.raw is not None:
            ad.layers["raw_counts"] = ad.raw.X.copy()
        else:
            ad.layers["raw_counts"] = ad.X.copy()
    if "feature_name" in ad.var.columns:
        ad.var_names = ad.var["feature_name"].astype(str)
        ad.var_names_make_unique()
    rc = ad.layers["raw_counts"]
    if sp.issparse(rc):
        vals = rc.data
    else:
        vals = np.asarray(rc).ravel()
    if vals.size:
        assert np.all(np.mod(vals[:min(5000, vals.size)], 1) == 0), "Raw counts are not integers!"
    if "in_tissue" in ad.obs:
        in_tissue = ad.obs["in_tissue"]
        if in_tissue.dtype == bool:
            mask = in_tissue
        else:
            mask = in_tissue.astype(int) == 1
        ad = ad[mask].copy()
    totals = np.array(ad.layers["raw_counts"].sum(axis=1)).ravel()
    ad = ad[totals > 0].copy()
    gene_totals = np.array(ad.layers["raw_counts"].sum(axis=0)).ravel()
    ad = ad[:, gene_totals > 0].copy()
    ad.X = ad.layers["raw_counts"].copy()
    sc.pp.normalize_total(ad, target_sum=1e4)
    sc.pp.log1p(ad)
    if hasattr(ad.X, "data"):
        ad.X.data = np.nan_to_num(ad.X.data)
    ad.layers["norm"] = ad.X.copy()
    return ad

In [14]:
# run function on all adatas
adatas_processed = [preprocess_visium(ad) for ad in adatas]

normalizing counts per cell
    finished (0:00:03)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
   

In [15]:
for ad in adatas_processed:
    if 'ec_subtype' not in ad.obs:
        majority = ad.obs['majority'].astype(str)
        ad.obs['ec_subtype'] = np.where(majority.isin(EC_SUBTYPES), majority, 'Other')
    if 'is_EC' not in ad.obs:
        ad.obs['is_EC'] = np.where(ad.obs['ec_subtype'] != 'Other', 'Endothelial', 'Other')

#### spatial plots of EC subtypes
based on 'majority' annotation
--> will use those as well for decoupleR

In [ ]:

#option of adding scalebar with always 1mm length:
def plot_spatial_grid(adatas, color_col, palette, filename,
                      img_key=None, size=1.2, show=True,
                      scalebar=True):
    ncols = 4
    nrows = math.ceil(len(adatas) / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(3*ncols, 3*nrows), facecolor='white')
    axes = np.array(axes).flatten()

    for i, ad in enumerate(adatas):
        lib_id = [k for k in ad.uns['spatial'] if k != 'is_single'][0]
        sample = ad.obs['sample_name'].iloc[0]
        age = ad.obs['age'].iloc[0]
        sc.pl.spatial(ad, color=color_col, library_id=lib_id, size=size,
                      palette=palette, img_key=img_key,
                      ax=axes[i], show=False, title=f'{sample} ({age}y)')
        if img_key is None:
            axes[i].set_facecolor('white')

        # ── Add scale bar ────────────────────────────────────────────
        if scalebar:
            sf = ad.uns['spatial'][lib_id]['scalefactors']

            if img_key is not None:
                spot_diam_px = sf['spot_diameter_fullres']
                if img_key == 'hires':
                    scale = sf.get('tissue_hires_scalef', 1.0)
                elif img_key == 'lowres':
                    scale = sf.get('tissue_lowres_scalef', 1.0)
                else:
                    scale = 1.0
                spot_diam_plot = spot_diam_px * scale
            else:
                spot_diam_plot = sf['spot_diameter_fullres']

            um_per_pixel = 55.0 / spot_diam_plot

            sb = ScaleBar(
            um_per_pixel, units='um',
            location='lower right',
            length_fraction=0.25,
            fixed_value=1000,         #(1000 µm)
            fixed_units='um',
            font_properties={'size': 0},
            scale_loc='none',
            label_loc='none',
            color='black',
            box_alpha=0,
            box_color='none',
            pad=0.1,
            sep=0,
            border_pad=0.3,
        )
            axes[i].add_artist(sb)

    for j in range(len(adatas), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(outputdir, filename), dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    plt.close()

In [ ]:
# EC subtypes — no H&E
plot_spatial_grid(adatas_processed, 'ec_subtype', PALETTE_SUBTYPES,
                  'EC_subtype_spatial_all.pdf')

# EC subtypes — with H&E (export only)
plot_spatial_grid(adatas_processed, 'ec_subtype', PALETTE_SUBTYPES,
                  'EC_subtype_spatial_HE.pdf', img_key='hires', show=False)

# Binary EC — no H&E
plot_spatial_grid(adatas_processed, 'is_EC', PALETTE_EC_BINARY,
                  'EC_endothelial_spatial_all.pdf')

# Binary EC — with H&E (export only)
plot_spatial_grid(adatas_processed, 'is_EC', PALETTE_EC_BINARY,
                  'EC_endothelial_spatial_HE.pdf', img_key='hires', show=False)

In [ ]:
#export with scalebars to plot
# Binary EC — no H&E, with scale bar
plot_spatial_grid(adatas_processed, 'is_EC', PALETTE_EC_BINARY,
                  'EC_endothelial_spatial_all_scalebar.pdf',
                  scalebar=True, show=False)

# Binary EC — with H&E, with scale bar
plot_spatial_grid(adatas_processed, 'is_EC', PALETTE_EC_BINARY,
                  'EC_endothelial_spatial_HE_scalebar.pdf',
                  img_key='hires', show=False, scalebar=True)

## run decoupler

#### use collectri network - as used in our manuscript - to include YAP TF network

In [21]:
#load network
collectri = dc.op.collectri(organism='human')

for ad in adatas_processed:
    sample_name = ad.obs['sample_name'].iloc[0]
    dc.mt.ulm(data=ad, net=collectri, layer='norm')
    score = dc.pp.get_obsm(adata=ad, key='score_ulm')
    ad.obsm['tf_scores'] = score.to_df()

    # Convenience column for spatial plotting
    if 'YAP1' in ad.obsm['tf_scores'].columns:
        ad.obs['YAP1_score'] = ad.obsm['tf_scores']['YAP1'].values

    print(f'{sample_name}: done – {ad.shape}')

print(f"\nAll {len(adatas_processed)} samples processed.")

body_abdomen_A: done – (1755, 19530)
body_back_A_a: done – (2418, 19579)
body_back_A_b: done – (2546, 21528)
body_breast_A: done – (2097, 20745)
body_inguinal_A_a: done – (1310, 20219)
body_inguinal_B: done – (1958, 20392)
body_pubis_A: done – (1632, 20735)
body_thigh_A: done – (1826, 20557)
face_cheek_A: done – (2036, 22105)
face_cheek_B: done – (1383, 22231)
face_forehead_A_a: done – (1163, 20530)
face_forehead_A_b: done – (1987, 21785)
face_forehead_B: done – (963, 21394)
face_glabella_A: done – (1577, 22490)
face_nose_A_a: done – (1552, 21909)
face_nose_A_b: done – (1642, 22238)
face_scalp_A: done – (1344, 21139)
face_temple_A: done – (927, 20985)
face_temple_A_a: done – (986, 21212)
face_temple_B_a: done – (1547, 21946)
face_temple_B_b: done – (1494, 21759)

All 21 samples processed.


In [ ]:
#prepare df_all (for plotting in barplot, vlnplot, ...)

import anndata

tf_frames = []
meta_frames = []

for ad in adatas_processed:
    tf = ad.obsm['tf_scores'].copy()
    tf.index = ad.obs_names

    meta = ad.obs[['age', 'sample_name', 'ec_subtype', 'majority',
                    'cell_type', 'donor_id', 'is_EC']].copy()
    meta.index = ad.obs_names

    tf_frames.append(tf)
    meta_frames.append(meta)

tf_all   = pd.concat(tf_frames)
meta_all = pd.concat(meta_frames)

#take dataframe from ad metadata
df_all = meta_all.copy()
df_all['age'] = df_all['age'].astype(int)
df_all['donor_label'] = df_all['sample_name'] + ' (' + df_all['age'].astype(str) + 'y)'

#add TF scores as columns (YAP1, TEAD1, TEAD4, WWTR1, …)
for col in tf_all.columns:
    df_all[col] = tf_all[col].values

#subset EC metadata
df_ec = df_all[df_all['ec_subtype'] != 'Other'].copy()

#set donor age as factor for linear plots
donor_order = (df_all[['donor_label', 'age']
               ].drop_duplicates()
                .sort_values('age')['donor_label'].tolist())
donor_order_ec = [d for d in donor_order if d in df_ec['donor_label'].values]

#
donor_age_map = dict(zip(df_all['donor_label'], df_all['age']))
unique_ages = sorted(df_all['age'].unique())
_cmap = LinearSegmentedColormap.from_list("age", ['#4b4e52', '#266c91', '#f0cf16'])
age_colors = {a: _cmap(i / max(1, len(unique_ages)-1))
              for i, a in enumerate(unique_ages)}
donor_palette = {d: age_colors[donor_age_map[d]] for d in donor_order}

print(f"df_all: {df_all.shape}  |  df_ec: {df_ec.shape}")
print(f"TF columns present: {[c for c in TF_GENES if c in df_all.columns]}")

#### plot scatter, mean YAP1 activity over age

In [ ]:
# ── Per-donor scatter (already uses mean — unchanged) ──
def plot_donor_scatter(df, genes, title_suffix, filename):
    fig, axes = plt.subplots(1, len(genes),
                             figsize=(3.5*len(genes), 3.5), facecolor='white')
    if len(genes) == 1:
        axes = [axes]

    for i, tf in enumerate(genes):
        ax = axes[i]
        dm = df.groupby(['sample_name', 'age'])[tf].mean().reset_index()

        for _, row in dm.iterrows():
            ax.scatter(row['age'], row[tf],
                       c=[age_colors[row['age']]],
                       s=60, edgecolor='black', zorder=3)

        slope, intercept, r_val, p_val, _ = stats.linregress(
            dm['age'], dm[tf])
        xl = np.linspace(dm['age'].min(), dm['age'].max(), 100)
        ax.plot(xl, slope*xl + intercept, 'k--', alpha=0.7)

        ax.set_xlabel('Age'); ax.set_ylabel(f'Mean {tf} score')
        ax.set_title(f'{tf} – {title_suffix}\n'
                     f'R²={r_val**2:.3f}, p={p_val:.2e}', fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(outputdir, filename), dpi=300, bbox_inches='tight')
    plt.show(); plt.close()

plot_donor_scatter(df_all, TF_GENES, 'All Cells', 'TF_scatter_all.pdf')
plot_donor_scatter(df_ec,  TF_GENES, 'EC Only',   'TF_scatter_EC.pdf')

#### plotting R2 in ECs vs other celltypes

In [ ]:
def _corr_by_annotation(df, col_name, min_spots=50):
    """Pearson r(YAP1, age) per category in `col_name`."""
    df_v = df.dropna(subset=[col_name, 'YAP1', 'age']).copy()
    df_v[col_name] = df_v[col_name].astype(str)
    df_v = df_v[df_v[col_name] != 'nan']
    results = []
    for ct in df_v[col_name].unique():
        sub = df_v[df_v[col_name] == ct]
        if len(sub) < min_spots:
            continue
        dm = sub.groupby(['sample_name', 'age'])['YAP1'].mean().reset_index()
        if len(dm) >= 3:
            r, p = stats.pearsonr(dm['age'], dm['YAP1'])
            results.append({'cell_type': ct, 'r': r, 'r2': r**2,
                            'p_value': p, 'n_spots': len(sub)})
    return pd.DataFrame(results).sort_values('r')

#majority with EC subtypes merged into one label
df_all['majority_merged'] = df_all['majority'].astype(str).replace(
    {s: 'Endothelial Cells' for s in EC_SUBTYPES}
)
df_merged_r = _corr_by_annotation(df_all, 'majority_merged')

if len(df_majority_r) and len(df_celltype_r) and len(df_merged_r):
    ec_ct_original = {'endothelial cell of vascular tree',
                      'endothelial cell of lymphatic vessel'}

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor='white')

    for ax, df_r, highlight, title in [
        (axes[0], df_majority_r,
         lambda ct: PALETTE_SUBTYPES.get(ct, 'gray'),
         'Majority (EC subtypes separate)'),
        (axes[1], df_celltype_r,
         lambda ct: 'red' if ct in ec_ct_original else 'gray',
         'Original annotation (EC in red)'),
        (axes[2], df_merged_r,
         lambda ct: 'red' if ct == 'Endothelial Cells' else 'gray',
         'Majority (EC merged)'),
    ]:
        cols = [highlight(ct) for ct in df_r['cell_type']]
        ax.barh(range(len(df_r)), df_r['r'], color=cols,
                edgecolor='black', linewidth=0.5, alpha=0.8)
        for i, (_, row) in enumerate(df_r.iterrows()):
            sig = ('***' if row['p_value'] < .001
                   else '**' if row['p_value'] < .01
                   else '*' if row['p_value'] < .05 else '')
            if sig:
                ax.text(row['r'] + 0.02 * np.sign(row['r']), i,
                        sig, va='center', fontsize=7)
        ax.set_yticks(range(len(df_r)))
        ax.set_yticklabels(df_r['cell_type'], fontsize=7)
        ax.axvline(0, color='black', linewidth=0.5)
        ax.set_xlabel('r (YAP1 ~ Age)')
        ax.set_title(title, fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(outputdir, 'YAP1_age_celltype_comparison.pdf'),
                dpi=300, bbox_inches='tight')
    plt.show(); plt.close()
else:
    print("Not enough data for cell-type correlation plot.")